# 2D and 3D Object Detection Tutorial

Make sure you fill in any place that says `YOUR CODE HERE` or `YOUR ANSWER HERE`. Do not delete any cells.

Fill in your details below:

In [ ]:
%matplotlib inline

---

# Object Detection Tutorial

- **Topic:** 2D detection, Non-Maximum Suppression, LiDAR point clouds, and 3D visualisation
- **Dataset:** KITTI Object Detection Benchmark

## Objectives

In this tutorial you will study the core building blocks of modern object detection pipelines. Starting from the fundamental representation of bounding boxes, you will build up to running a 2D detector, suppressing duplicate predictions with Non-Maximum Suppression, and finally working with LiDAR points and 3D bounding boxes. By the end of this tutorial, you will be able to:

- represent and manipulate 2D and 3D bounding boxes.
- compute Intersection over Union (IoU) between bounding boxes.
- parse KITTI ground truth label files.
- run inference with a pre-trained YOLO detector and parse its outputs.
- implement Non-Maximum Suppression (NMS) from scratch.
- project KITTI Velodyne LiDAR points into the camera image.
- isolate a pedestrian point cloud from a 2D pedestrian bounding box.
- cluster pedestrian LiDAR points and visualise a 3D box in the raw point cloud.

<div class="alert alert-info">
Note: Written questions should be answered concisely — a few sentences or bullet points suffice.
</div>

<div class="alert alert-warning">
Warning: Do not delete any cells. Do not use all-UPPERCASE variable names.
</div>

---

## Dataset: KITTI Object Detection Benchmark

This tutorial uses the **KITTI Object Detection Benchmark**, a real-world dataset collected from a vehicle driving through Karlsruhe, Germany. The vehicle is equipped with cameras and a LiDAR sensor.

The next cell automatically downloads and extracts the KITTI object-detection files needed for this notebook:

- `data_object_image_2.zip` — left color camera images
- `data_object_label_2.zip` — object labels
- `data_object_calib.zip` — camera/LiDAR calibration
- `data_object_velodyne.zip` — Velodyne LiDAR point clouds

<div class="alert alert-warning">
The image and Velodyne zip files are large. This can take a long time and requires a lot of disk space. If the data is already present, the cell skips downloading it.
</div>

After the cell runs, the folder structure should be:

```
kitti/
├── training/
│   ├── image_2/        ← left camera images  (*.png)
│   ├── label_2/        ← ground truth labels (*.txt)
│   ├── calib/          ← calibration files   (*.txt)
│   └── velodyne/       ← LiDAR point clouds  (*.bin)
```



In [ ]:
# CELL REF: C0.0a
# Automatic KITTI Object Detection download and setup

import os
import urllib.request
import zipfile
from pathlib import Path

# Dataset location used by the rest of the notebook
DATA_ROOT = Path("kitti")
KITTI_ROOT = str(DATA_ROOT / "training")

IMAGE_DIR = os.path.join(KITTI_ROOT, "image_2")
LABEL_DIR = os.path.join(KITTI_ROOT, "label_2")
CALIB_DIR = os.path.join(KITTI_ROOT, "calib")
VELODYNE_DIR = os.path.join(KITTI_ROOT, "velodyne")

# Official KITTI object-detection files hosted on the KITTI S3 mirror.
# The image and Velodyne archives are large, so this cell skips files that already exist.
kitti_files = {
    "data_object_image_2.zip": "image_2",
    "data_object_label_2.zip": "label_2",
    "data_object_calib.zip": "calib",
    "data_object_velodyne.zip": "velodyne",
}

base_url = "https://s3.eu-central-1.amazonaws.com/avg-kitti"
zip_dir = DATA_ROOT / "zips"
zip_dir.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)


def download_with_progress(url, destination):
    """Download a URL to destination with a simple progress display."""
    destination = Path(destination)

    def reporthook(block_num, block_size, total_size):
        if total_size <= 0:
            return
        downloaded = block_num * block_size
        percent = min(100, downloaded * 100 / total_size)
        mb_done = downloaded / (1024 ** 2)
        mb_total = total_size / (1024 ** 2)
        print(f"\rDownloading {destination.name}: {percent:5.1f}% "
              f"({mb_done:,.1f}/{mb_total:,.1f} MB)", end="")

    urllib.request.urlretrieve(url, destination, reporthook=reporthook)
    print()


def extract_zip(zip_path, extract_to):
    """Extract a KITTI zip file."""
    print(f"Extracting {zip_path.name} ...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_to)
    print(f"Done extracting {zip_path.name}")


# Download and extract required KITTI parts
for zip_name, expected_folder in kitti_files.items():
    expected_path = Path(KITTI_ROOT) / expected_folder
    zip_path = zip_dir / zip_name
    url = f"{base_url}/{zip_name}"

    if expected_path.exists() and any(expected_path.iterdir()):
        print(f"Found {expected_path}; skipping {zip_name}")
        continue

    if not zip_path.exists():
        print(f"Downloading {zip_name} from KITTI mirror...")
        download_with_progress(url, zip_path)
    else:
        print(f"Found existing archive {zip_path}; skipping download")

    extract_zip(zip_path, DATA_ROOT)

# Collect all sample IDs (6-digit filenames without extension)
if not os.path.isdir(IMAGE_DIR):
    raise FileNotFoundError(f"IMAGE_DIR not found: {IMAGE_DIR}")

sample_ids = sorted(
    f.replace(".png", "")
    for f in os.listdir(IMAGE_DIR)
    if f.endswith(".png")
)

print("\nKITTI setup complete.")
print(f"Found {len(sample_ids)} images in {IMAGE_DIR}")
print(f"First few sample IDs: {sample_ids[:5]}")
print("Label dir:   ", LABEL_DIR)
print("Calib dir:   ", CALIB_DIR)
print("Velodyne dir:", VELODYNE_DIR)



## Setup

Run the cell below to import all libraries used throughout this tutorial.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from copy import copy

np.random.seed(42)

from IPython.core.display import HTML
HTML("""
<style>
.output_png {
    display: table-cell;
    text-align: center;
    vertical-align: middle;
}
</style>
""")

### Helper: loading a KITTI image

We provide `load_image` to load a KITTI image as a NumPy array. You will use this throughout the tutorial.

In [ ]:
# CELL REF: C0.3a
# Helper functions — do not modify

def load_image(sample_id):
    """Load a KITTI left camera image as a NumPy uint8 array (H, W, 3)."""
    path = os.path.join(IMAGE_DIR, sample_id + ".png")
    return np.array(Image.open(path).convert("RGB"))

# Quick check: display the first image
img = load_image(sample_ids[0])
print(f"Image shape: {img.shape}  (H x W x C)")
plt.figure(figsize=(12, 4))
plt.imshow(img)
plt.axis("off")
plt.title(f"Sample ID: {sample_ids[0]}")
plt.tight_layout()
plt.show()

---

## Part 1 - Bounding Boxes and IoU

A **bounding box** is the fundamental data structure in object detection. It is a rectangle that tightly encloses a detected object in the image. There are two common formats:

- **Corner format**: `[x1, y1, x2, y2]` — top-left and bottom-right pixel coordinates.
- **Centre format**: `[cx, cy, w, h]` — centre coordinates, width, and height.

Both formats represent the same rectangle. Different detectors and evaluation tools use different conventions, so being able to convert between them fluently is essential.

The KITTI label files store boxes in corner format. YOLO (which you will use in Part 2) outputs boxes in centre format. You will need both.

---

## Exercise 1.1 - Bounding Box Representations

Implement `corners_to_centre` and `centre_to_corners` to convert between the two formats.
Both functions receive a NumPy array of shape `(4,)` and must return a NumPy array of shape `(4,)`.

<div class="alert alert-success">
Hint: The centre x is the average of x1 and x2. The width is simply x2 - x1.
</div>

In [ ]:
# CELL REF: C1.1a

def corners_to_centre(box):
    """
    Convert bounding box from corner format to centre format.
    Input:  box = [x1, y1, x2, y2]
    Output: [cx, cy, w, h]
    """
    box = np.asarray(box, dtype=float)
    x1, y1, x2, y2 = box
    cx = (x1 + x2) / 2.0
    cy = (y1 + y2) / 2.0
    w = x2 - x1
    h = y2 - y1
    return np.array([cx, cy, w, h], dtype=float)


def centre_to_corners(box):
    """
    Convert bounding box from centre format to corner format.
    Input:  box = [cx, cy, w, h]
    Output: [x1, y1, x2, y2]
    """
    box = np.asarray(box, dtype=float)
    cx, cy, w, h = box
    x1 = cx - w / 2.0
    y1 = cy - h / 2.0
    x2 = cx + w / 2.0
    y2 = cy + h / 2.0
    return np.array([x1, y1, x2, y2], dtype=float)


# --- Quick sanity check ---
box_corners = np.array([10., 20., 50., 80.])
box_centre  = corners_to_centre(box_corners)
print(f"Corner format:   {box_corners}")
print(f"Centre format:   {box_centre}   (expected [30. 50. 40. 60.])")
print(f"Back to corners: {centre_to_corners(box_centre)}   (should match original)")

In [ ]:
# CELL REF: C1.1b

box = np.array([10., 20., 50., 80.])
centre = corners_to_centre(box)
assert centre.shape == (4,), "Output shape must be (4,)"
assert np.allclose(centre, [30., 50., 40., 60.]), f"Expected [30. 50. 40. 60.], got {centre}"
assert np.allclose(centre_to_corners(centre), box), "Round-trip conversion failed"
print("Exercise 1.1 passed.")

---

## Exercise 1.2 - Parsing KITTI Label Files

Each KITTI label file (one per image) contains one object per line. Each line has 15 space-separated fields:

```
type  truncated  occluded  alpha  x1  y1  x2  y2  h3d  w3d  l3d  x3d  y3d  z3d  yaw
```

For now we care about:
- **type** (field 0): class name, e.g. `'Car'`, `'Pedestrian'`, `'Cyclist'`, `'DontCare'`
- **x1, y1, x2, y2** (fields 4–7): 2D bounding box in corner format (pixels)

Implement `parse_kitti_labels` to read a label file and return a list of `(class_name, box)` tuples, where `box` is a NumPy array of shape `(4,)` in corner format. Skip lines where the class is `'DontCare'`.

<div class="alert alert-info">
Note: You can open a label file with <code>open(path).readlines()</code>. Each line can be split with <code>.split()</code>.
</div>

In [ ]:
# CELL REF: C1.2a

def parse_kitti_labels(sample_id):
    """
    Parse the KITTI label file for a given sample_id.
    Returns a list of (class_name, box) tuples.
      class_name: str  (e.g. 'Car', 'Pedestrian')
      box:        np.array of shape (4,) in corner format [x1, y1, x2, y2]
    Skip objects with class 'DontCare'.
    """
    path = os.path.join(LABEL_DIR, sample_id + ".txt")
    labels = []

    with open(path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 8:
                continue
            class_name = parts[0]
            if class_name == "DontCare":
                continue
            box = np.array([float(v) for v in parts[4:8]], dtype=float)
            labels.append((class_name, box))

    return labels


# --- Test on the first sample ---
labels = parse_kitti_labels(sample_ids[0])
print(f"Sample ID: {sample_ids[0]}  —  {len(labels)} objects (DontCare excluded)")
for cls, box in labels:
    print(f"  {cls:12s}  box={box}")

In [ ]:
# CELL REF: C1.2b

labels = parse_kitti_labels(sample_ids[0])
assert isinstance(labels, list), "parse_kitti_labels must return a list"
assert len(labels) > 0, "Expected at least one labelled object (non-DontCare)"
cls, box = labels[0]
assert isinstance(cls, str), "Class name must be a string"
assert isinstance(box, np.ndarray) and box.shape == (4,), "Box must be np.array of shape (4,)"
assert not any(c == "DontCare" for c, _ in labels), "DontCare objects must be excluded"
print(f"Exercise 1.2 passed — {len(labels)} objects found.")

---

## Exercise 1.3 - Visualising Ground Truth Boxes

Implement `visualise_gt_boxes` to draw the ground truth bounding boxes on a KITTI image. Use a different colour for each class. Then visualise the ground truth for the first 4 images in the dataset.

<div class="alert alert-success">
Hint: Use a dictionary to map class names to colours, e.g. <code>{{'Car': 'red', 'Pedestrian': 'cyan', 'Cyclist': 'yellow'}}</code>. Use <code>matplotlib.patches.Rectangle</code> to draw boxes.
</div>

In [ ]:
# CELL REF: C1.3a

# Colour map for KITTI classes
CLASS_COLORS = {
    'Car':        'red',
    'Pedestrian': 'cyan',
    'Cyclist':    'yellow',
    'Van':        'orange',
    'Truck':      'magenta',
    'Misc':       'white',
}

def visualise_gt_boxes(sample_id, ax=None):
    """
    Draw ground truth bounding boxes on the KITTI image for sample_id.
    Uses CLASS_COLORS for colour-coding by class.
    """
    image  = load_image(sample_id)
    labels = parse_kitti_labels(sample_id)

    if ax is None:
        fig, ax = plt.subplots(1, figsize=(12, 4))

    ax.imshow(image)

    for cls, box in labels:
        x1, y1, x2, y2 = box
        color = CLASS_COLORS.get(cls, 'lime')
        rect = patches.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            linewidth=2,
            edgecolor=color,
            facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(
            x1,
            max(y1 - 3, 0),
            cls,
            color='black',
            fontsize=9,
            bbox=dict(facecolor=color, alpha=0.75, edgecolor='none')
        )

    ax.set_title(f"Ground truth — sample {sample_id}  ({len(labels)} objects)")
    ax.axis("off")


# --- Visualise the first 4 samples ---
fig, axes = plt.subplots(2, 2, figsize=(18, 8))
for ax, sid in zip(axes.flat, sample_ids[:4]):
    visualise_gt_boxes(sid, ax=ax)
plt.tight_layout()
plt.show()

---

## Exercise 1.4 - Intersection over Union

**Intersection over Union (IoU)** measures the overlap between two bounding boxes:

$$\text{IoU} = \frac{\text{Area of Intersection}}{\text{Area of Union}}$$

IoU = 0 means the boxes do not overlap at all. IoU = 1 means they are identical. It is used both to evaluate how well a predicted box matches a ground truth box, and as the suppression criterion in NMS (Part 3).

Implement `compute_iou` which takes two boxes in corner format `[x1, y1, x2, y2]` and returns a scalar float.

<div class="alert alert-success">
Hint: The intersection rectangle's top-left is <code>(max(x1_a, x1_b), max(y1_a, y1_b))</code> and bottom-right is <code>(min(x2_a, x2_b), min(y2_a, y2_b))</code>. If this does not form a valid rectangle (width or height ≤ 0), the intersection area is 0.
</div>

In [ ]:
# CELL REF: C1.4a

def compute_iou(box_a, box_b):
    """
    Compute Intersection over Union between two boxes in corner format.
    box_a, box_b: np.array of shape (4,)  [x1, y1, x2, y2]
    Returns: float in [0, 1]
    """
    box_a = np.asarray(box_a, dtype=float)
    box_b = np.asarray(box_b, dtype=float)

    x_left = max(box_a[0], box_b[0])
    y_top = max(box_a[1], box_b[1])
    x_right = min(box_a[2], box_b[2])
    y_bottom = min(box_a[3], box_b[3])

    inter_w = max(0.0, x_right - x_left)
    inter_h = max(0.0, y_bottom - y_top)
    intersection = inter_w * inter_h

    area_a = max(0.0, box_a[2] - box_a[0]) * max(0.0, box_a[3] - box_a[1])
    area_b = max(0.0, box_b[2] - box_b[0]) * max(0.0, box_b[3] - box_b[1])
    union = area_a + area_b - intersection

    if union <= 0:
        return 0.0
    return float(intersection / union)


# --- Sanity checks ---
b1 = np.array([0.,  0., 10., 10.])
b2 = np.array([0.,  0., 10., 10.])   # identical
b3 = np.array([5.,  5., 15., 15.])   # 25px² overlap
b4 = np.array([20., 20., 30., 30.])  # no overlap

print(f"Identical boxes:  IoU = {compute_iou(b1, b2):.4f}  (expected 1.0000)")
print(f"Partial overlap:  IoU = {compute_iou(b1, b3):.4f}  (expected 0.1429)")
print(f"No overlap:       IoU = {compute_iou(b1, b4):.4f}  (expected 0.0000)")

In [ ]:
# CELL REF: C1.4b

assert np.isclose(compute_iou(np.array([0.,0.,10.,10.]), np.array([0.,0.,10.,10.])), 1.0)
assert np.isclose(compute_iou(np.array([0.,0.,10.,10.]), np.array([20.,20.,30.,30.])), 0.0)
assert np.isclose(compute_iou(np.array([0.,0.,10.,10.]), np.array([5.,5.,15.,15.])), 25/175, atol=1e-4)
print("Exercise 1.4 passed.")

**Q1.4a**: What does IoU = 0 vs IoU = 1 mean physically for two bounding boxes?

YOUR ANSWER HERE

**Q1.4b**: Why is IoU preferred over raw pixel overlap area as a matching criterion?

YOUR ANSWER HERE

---

## Part 2 - 2D Object Detection with YOLO

A 2D object detector takes an image as input and outputs a set of predicted bounding boxes, each paired with a class label and a confidence score. In this part you will use **YOLOv5**, a popular single-stage detector, pre-trained on the COCO dataset.

A key property of these detectors is that they produce **many overlapping predictions** for the same object. The network evaluates thousands of candidate locations simultaneously and predicts an object at each one independently — most with low confidence, but many with high confidence around the same real object. Filtering these duplicates is the job of NMS, which you will implement in Part 3.

<div class="alert alert-warning">
Warning: YOLOv5 is pre-trained on COCO, not KITTI. COCO class names differ slightly from KITTI (e.g. COCO uses <code>'car'</code> and <code>'person'</code>). You do not need to retrain the model — the goal here is to understand the detector output format.
</div>

---

### Setup for Part 2

The cell below installs and loads YOLOv5 using `torch.hub`. This requires an internet connection and will download model weights (~14 MB) on the first run.

<div class="alert alert-info">
Note on PyTorch: <code>torch.hub.load</code> downloads a model from a GitHub repository and returns it ready to use. <code>model.eval()</code> switches the model to inference mode (disables dropout and batch normalisation training behaviour). <code>torch.no_grad()</code> tells PyTorch not to track gradients, which saves memory and speeds up inference.
</div>

In [ ]:
# CELL REF: C2.0a

import torch

# torch.hub.load downloads YOLOv5 from GitHub and returns a PyTorch model object.
# 'yolov5s' is the small variant — fast and sufficient for this tutorial.
# trust_repo=True is required for newer versions of PyTorch hub.
model_yolo = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True, trust_repo=True)

# Switch to evaluation / inference mode.
# Always do this before running inference — without it the model behaves differently
# due to dropout layers being active during training.
model_yolo.eval()

print("YOLOv5s loaded successfully.")
print("YOLOv5s loaded successfully.")

# Print first 10 class names
first_10 = dict(list(model_yolo.names.items())[:10])

print("Model class names (first 10):")
print(first_10)

---

## Exercise 2.1 - Running YOLO on a KITTI Image

Run the YOLO model on the first KITTI image and store the result in `yolo_result`. Then print the shape and content of the raw prediction tensor.

<div class="alert alert-info">
Note on running YOLOv5: pass the image file path directly to the model as a string: <code>result = model_yolo(image_path)</code>. The result object has a <code>.xyxy</code> attribute which is a list of tensors (one per image). Each tensor has shape <code>(N, 6)</code> where each row is <code>[x1, y1, x2, y2, confidence, class_id]</code>. Use <code>.xyxy[0]</code> to get predictions for the first (and only) image. Convert to NumPy with <code>.numpy()</code>.
</div>

In [ ]:
# CELL REF: C2.1a

# Path to the first KITTI image
first_image_path = os.path.join(IMAGE_DIR, sample_ids[0] + ".png")

# 1. Run YOLO on the first image
# 2. Extract raw predictions as a NumPy array with columns:
#    [x1, y1, x2, y2, confidence, class_id]
yolo_result = model_yolo(first_image_path)
raw_preds = yolo_result.xyxy[0].detach().cpu().numpy()

print(f"Raw prediction array shape: {raw_preds.shape}  (N detections x 6 columns)")
print("Columns: [x1, y1, x2, y2, confidence, class_id]")
print("\nFirst 5 predictions:")
print(raw_preds[:5])

In [ ]:
# CELL REF: C2.1b

assert yolo_result is not None, "yolo_result must be set"
assert isinstance(raw_preds, np.ndarray), "raw_preds must be a NumPy array"
assert raw_preds.ndim == 2 and raw_preds.shape[1] == 6, "raw_preds must have shape (N, 6)"
assert raw_preds[:, 4].max() <= 1.0, "Confidence scores must be in [0, 1]"
print(f"Total raw predictions on first image: {len(raw_preds)}")
print("Exercise 2.1 passed.")

**Q2.1a**: Why does YOLO produce many predictions for the same object rather than a single clean box?

YOUR ANSWER HERE

---

## Exercise 2.2 - Parsing YOLO Detections

Implement `parse_yolo_detections` to convert the raw `(N, 6)` array into a clean list of `(box, class_name, score)` tuples, where `box` is a NumPy array in corner format. Use `model_yolo.names` to map class IDs to class name strings.

Then implement `filter_by_confidence` to keep only detections above a given score threshold.

<div class="alert alert-info">
Note: <code>model_yolo.names</code> is a list of class name strings. <code>model_yolo.names[int(class_id)]</code> gives the name for a given class ID.
</div>

In [ ]:
# CELL REF: C2.2a

def parse_yolo_detections(raw_preds):
    """
    Convert YOLO raw prediction array to a list of (box, class_name, score) tuples.
    raw_preds: np.array of shape (N, 6) — columns: [x1, y1, x2, y2, confidence, class_id]
    Returns: list of (box, class_name, score)
      box:        np.array of shape (4,) in corner format [x1, y1, x2, y2]
      class_name: str
      score:      float
    """
    detections = []
    names = model_yolo.names

    for row in np.asarray(raw_preds):
        box = row[:4].astype(float)
        score = float(row[4])
        class_id = int(row[5])
        if isinstance(names, dict):
            class_name = names.get(class_id, str(class_id))
        else:
            class_name = names[class_id]
        detections.append((box, class_name, score))

    return detections


def filter_by_confidence(detections, threshold):
    """
    Keep only detections with score >= threshold.
    detections: list of (box, class_name, score)
    Returns: filtered list
    """
    return [det for det in detections if det[2] >= threshold]


all_detections = parse_yolo_detections(raw_preds)
print(f"Total detections parsed: {len(all_detections)}")
if len(all_detections) > 0:
    print(f"Example: class={all_detections[0][1]}, score={all_detections[0][2]:.3f}, box={all_detections[0][0]}")
else:
    print("No YOLO detections found for this image.")

In [ ]:
# CELL REF: C2.2b

assert isinstance(all_detections, list), "parse_yolo_detections must return a list"
assert len(all_detections) == len(raw_preds), "Length must match number of raw predictions"
box, cls, score = all_detections[0]
assert isinstance(box, np.ndarray) and box.shape == (4,), "Box must be np.array of shape (4,)"
assert isinstance(cls, str), "Class name must be a string"
assert 0.0 <= score <= 1.0, "Score must be in [0, 1]"

filtered = filter_by_confidence(all_detections, 0.5)
assert all(s >= 0.5 for _, _, s in filtered), "All filtered detections must have score >= threshold"
print(f"Detections above 0.5 confidence: {len(filtered)}")
print("Exercise 2.2 passed.")

---

## Exercise 2.3 - Visualising Detections at Different Confidence Thresholds

Implement `visualise_detections` to draw the filtered YOLO detections on a KITTI image, annotating each box with its class name and confidence score.

Then, for the first 3 KITTI images, display the detections side by side for confidence thresholds of **0.25**, **0.5**, and **0.75**.

<div class="alert alert-success">
Hint: Use <code>ax.text(x1, y1 - 5, f"{cls} {score:.2f}")</code> to place the label just above the box.
</div>

In [ ]:
# CELL REF: C2.3a

def visualise_detections(sample_id, detections, ax=None, title=""):
    """
    Draw YOLO detections on the KITTI image for sample_id.
    detections: list of (box, class_name, score)
    """
    image = load_image(sample_id)

    if ax is None:
        fig, ax = plt.subplots(1, figsize=(12, 4))

    ax.imshow(image)

    for box, class_name, score in detections:
        x1, y1, x2, y2 = box
        rect = patches.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            linewidth=2,
            edgecolor='red',
            facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(
            x1,
            max(y1 - 3, 0),
            f"{class_name} {score:.2f}",
            color='white',
            fontsize=8,
            bbox=dict(facecolor='red', alpha=0.75, edgecolor='none')
        )

    ax.set_title(title if title else f"{len(detections)} detections")
    ax.axis("off")


# --- Visualise for first 3 images at 3 thresholds ---
thresholds = [0.25, 0.5, 0.75]
fig, axes = plt.subplots(3, 3, figsize=(20, 12))

for row, sid in enumerate(sample_ids[:3]):
    img_path = os.path.join(IMAGE_DIR, sid + ".png")
    result   = model_yolo(img_path)
    preds    = result.xyxy[0].detach().cpu().numpy()
    dets     = parse_yolo_detections(preds)

    for col, thresh in enumerate(thresholds):
        filtered = filter_by_confidence(dets, thresh)
        visualise_detections(sid, filtered, ax=axes[row][col],
                             title=f"Sample {sid} | threshold={thresh} | {len(filtered)} dets")

plt.tight_layout()
plt.show()

**Q2.3a**: What happens to the number of detections as you increase the confidence threshold? How does this relate to the precision-recall trade-off?

YOUR ANSWER HERE

**Q2.3b**: Even at a high confidence threshold you may still see multiple boxes around the same object. What would be needed to fix this?

YOUR ANSWER HERE

---

## Part 3 - Non-Maximum Suppression

As you observed in Part 2, a detector produces many overlapping boxes for the same object. **Non-Maximum Suppression (NMS)** is the standard post-processing step used to remove these duplicates while keeping the best detection for each object.

The algorithm works as follows:

1. Sort all detections by confidence score in **descending** order.
2. Take the highest-scoring detection and add it to the output (keep it).
3. Remove all remaining detections whose IoU with the kept box is **≥ iou_threshold**.
4. Repeat from step 2 on the remaining detections until none are left.

The `iou_threshold` controls how aggressively overlapping boxes are suppressed. A lower threshold suppresses more; a higher threshold allows more boxes to survive.

---

## Exercise 3.1 - Implementing NMS from Scratch

Implement `nms` which takes a list of detections and an IoU threshold and returns the surviving detections after suppression.

<div class="alert alert-warning">
Warning: Do not use any external NMS function (e.g. from torchvision or ultralytics). You must implement the algorithm yourself using your <code>compute_iou</code> from Part 1.
</div>

<div class="alert alert-success">
Hint: Maintain a list of indices that are still "active". At each step, take the index with the highest score, add it to your result, then mark as inactive any index whose box has IoU ≥ iou_threshold with the kept box.
</div>

In [ ]:
# CELL REF: C3.1a

def nms(detections, iou_threshold=0.5):
    """
    Apply greedy Non-Maximum Suppression to a list of detections.

    detections:    list of (box, class_name, score)
                   box in corner format [x1, y1, x2, y2]
    iou_threshold: float — boxes with IoU >= this value are suppressed

    Returns: filtered list of (box, class_name, score)
    """
    if len(detections) == 0:
        return []

    remaining = sorted(detections, key=lambda det: det[2], reverse=True)
    kept = []

    while remaining:
        best = remaining.pop(0)
        kept.append(best)

        new_remaining = []
        for det in remaining:
            if compute_iou(best[0], det[0]) < iou_threshold:
                new_remaining.append(det)
        remaining = new_remaining

    return kept


# --- Quick test on synthetic boxes ---
test_dets = [
    (np.array([10., 10., 60., 60.]), 'car', 0.95),
    (np.array([12., 12., 62., 62.]), 'car', 0.80),    # nearly identical → suppressed
    (np.array([14., 14., 64., 64.]), 'car', 0.72),    # also overlaps → suppressed
    (np.array([200., 200., 280., 280.]), 'car', 0.88), # far away → survives
]

result = nms(test_dets, iou_threshold=0.5)
print(f"Before NMS: {len(test_dets)} boxes")
print(f"After NMS:  {len(result)} boxes  (expected 2)")
for box, cls, score in result:
    print(f"  {cls}  score={score:.2f}  box={box}")

In [ ]:
# CELL REF: C3.1b

# Two identical boxes → only the higher-scoring one survives
dets_identical = [
    (np.array([0., 0., 10., 10.]), 'car', 0.9),
    (np.array([0., 0., 10., 10.]), 'car', 0.8),
]
r = nms(dets_identical, iou_threshold=0.5)
assert len(r) == 1, f"Expected 1 surviving box, got {len(r)}"
assert r[0][2] == 0.9, "The highest-scoring box must be kept"

# Two non-overlapping boxes → both survive
dets_separate = [
    (np.array([0.,  0.,  10., 10.]), 'car', 0.9),
    (np.array([50., 50., 60., 60.]), 'car', 0.8),
]
r2 = nms(dets_separate, iou_threshold=0.5)
assert len(r2) == 2, f"Expected 2 surviving boxes, got {len(r2)}"

print("Exercise 3.1 passed.")

**Q3.1a**: Why must detections be sorted by confidence score before suppression begins? What would go wrong if you processed them in arbitrary order?

YOUR ANSWER HERE

---

## Exercise 3.2 - Before and After NMS on KITTI Images

Apply your NMS to YOLO detections on the first 4 KITTI images. For each image, show the detections **before** and **after** NMS side by side. Use a confidence threshold of 0.25 before NMS, and an NMS IoU threshold of 0.5.

<div class="alert alert-info">
Note: Run the YOLO model once per image and reuse the detections for both the before and after columns.
</div>

In [ ]:
# CELL REF: C3.2a

fig, axes = plt.subplots(4, 2, figsize=(18, 20))

for row, sid in enumerate(sample_ids[:4]):
    img_path = os.path.join(IMAGE_DIR, sid + ".png")

    # Run YOLO and parse
    result = model_yolo(img_path)
    preds  = result.xyxy[0].detach().cpu().numpy()
    dets   = parse_yolo_detections(preds)

    # Apply confidence threshold
    dets_conf = filter_by_confidence(dets, threshold=0.25)

    visualise_detections(
        sid,
        dets_conf,
        ax=axes[row][0],
        title=f"Before NMS — {len(dets_conf)} detections"
    )

    dets_nms = nms(dets_conf, iou_threshold=0.5)

    visualise_detections(
        sid,
        dets_nms,
        ax=axes[row][1],
        title=f"After NMS — {len(dets_nms)} detections"
    )

plt.tight_layout()
plt.show()

---

## Exercise 3.3 - Effect of the NMS IoU Threshold

Using the first KITTI image, sweep the NMS IoU threshold from 0.1 to 0.9 in steps of 0.1. For each threshold, count how many detections survive. Plot the surviving count against the threshold.

Start from the confidence-filtered detections (threshold = 0.25) and apply NMS on top.

In [ ]:
# CELL REF: C3.3a

# Run detector once on the first image
img_path_0 = os.path.join(IMAGE_DIR, sample_ids[0] + ".png")
result_0   = model_yolo(img_path_0)
dets_0     = parse_yolo_detections(result_0.xyxy[0].detach().cpu().numpy())
dets_0_conf = filter_by_confidence(dets_0, threshold=0.25)

thresholds = np.arange(0.1, 1.0, 0.1)
surviving_counts = []

for threshold in thresholds:
    dets_after_nms = nms(dets_0_conf, iou_threshold=float(threshold))
    surviving_counts.append(len(dets_after_nms))

plt.figure(figsize=(7, 4))
plt.plot(thresholds, surviving_counts, marker='o')
plt.xlabel("NMS IoU threshold")
plt.ylabel("Surviving detections")
plt.title(f"Effect of NMS IoU threshold — sample {sample_ids[0]}")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# CELL REF: C3.3b

assert len(surviving_counts) == len(thresholds), "One count per threshold value required"
assert surviving_counts[0] <= surviving_counts[-1], "Higher IoU threshold should suppress less (more boxes survive)"
print("Exercise 3.3 passed.")

**Q3.3a**: When would a very low NMS IoU threshold hurt recall? Give a concrete example using KITTI scene content.

YOUR ANSWER HERE

**Q3.3b**: The NMS above processes all class predictions together. Can you think of a case in a KITTI scene where this would incorrectly suppress a valid detection? How would you fix it?

YOUR ANSWER HERE

---

## Conclusion — Parts 1–3

In these three parts you have:

- represented bounding boxes in corner and centre formats and converted between them.
- parsed KITTI ground truth label files and visualised objects on real driving images.
- implemented IoU from scratch and verified it on known cases.
- run a pre-trained YOLOv5 detector on KITTI images and parsed its raw multi-box output.
- understood the confidence threshold as a precision-recall trade-off.
- implemented greedy NMS from scratch and studied the effect of its IoU threshold on a real dataset.

Parts 4 and 5 will extend this pipeline to 3D bounding boxes using KITTI calibration data, and to quantitative evaluation with mAP.

---

# Part 4 - LiDAR Pedestrian Extraction, Clustering, and 3D Visualisation

This section shows how to detect and segment object in 3D. In this section you will

1. load KITTI Velodyne point clouds and calibration files;
2. project raw LiDAR points into the camera image;
3. keep only points that fall inside a pedestrian 2D bounding box;
4. automatically cluster the points so the pedestrian remains;
5. visualise the selected pedestrian cluster in the image;
6. add a 3D bounding box around that pedestrian inside the raw point cloud.

Run the previous setup cells first so `KITTI_ROOT`, `IMAGE_DIR`, `LABEL_DIR`, `CALIB_DIR`, `sample_ids`, and `load_image` exist.


In [ ]:
# CELL REF: C4.0a
# Extra imports for LiDAR pedestrian extraction

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

try:
    from sklearn.cluster import DBSCAN
except ImportError as exc:
    raise ImportError("Install scikit-learn first: pip install scikit-learn") from exc

try:
    import open3d as o3d
    HAS_OPEN3D = True
except ImportError:
    HAS_OPEN3D = False
    print("open3d is not installed. 3D visualisation will use Plotly fallback where possible.")

try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False
    print("plotly is not installed. Install it with: pip install plotly")

VELODYNE_DIR = os.path.join(KITTI_ROOT, "velodyne")
print("Velodyne directory:", VELODYNE_DIR)
print("Exists:", os.path.isdir(VELODYNE_DIR))


These are some provided helper functions.

In [ ]:
# CELL REF: C4.1a
# KITTI LiDAR, labels, calibration, and projection helpers

def read_velodyne_bin(sample_id):
    """Load KITTI Velodyne .bin file as Nx4 array: [x, y, z, reflectance]."""
    path = os.path.join(VELODYNE_DIR, sample_id + ".bin")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Velodyne file not found: {path}")
    return np.fromfile(path, dtype=np.float32).reshape(-1, 4)


def parse_kitti_labels_full(sample_id):
    """Parse full KITTI labels, including 2D and 3D fields."""
    path = os.path.join(LABEL_DIR, sample_id + ".txt")
    objects = []

    with open(path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 15:
                continue
            objects.append({
                "type": parts[0],
                "truncated": float(parts[1]),
                "occluded": int(parts[2]),
                "alpha": float(parts[3]),
                "bbox": np.array([float(x) for x in parts[4:8]], dtype=np.float32),
                "dimensions_hwl": np.array([float(x) for x in parts[8:11]], dtype=np.float32),
                "location_xyz_camera": np.array([float(x) for x in parts[11:14]], dtype=np.float32),
                "rotation_y": float(parts[14]),
            })
    return objects


def read_calib(sample_id):
    """Read KITTI calibration file and reshape P2, R0_rect, Tr_velo_to_cam."""
    path = os.path.join(CALIB_DIR, sample_id + ".txt")
    calib = {}

    with open(path, "r") as f:
        for line in f:
            if ":" not in line:
                continue
            key, values = line.split(":", 1)
            values = np.array([float(v) for v in values.split()], dtype=np.float64)
            calib[key] = values

    calib["P2"] = calib["P2"].reshape(3, 4)
    calib["R0_rect"] = calib["R0_rect"].reshape(3, 3)
    calib["Tr_velo_to_cam"] = calib["Tr_velo_to_cam"].reshape(3, 4)
    return calib


def project_velodyne_to_image(points_velo, calib):
    """
    Project Velodyne points into KITTI camera-2 image coordinates.

    Returns:
      uv: projected pixels for points in front of the camera, shape Mx2
      depth: camera depth for projected points, shape M
      original_indices: indices mapping uv rows back to original points_velo rows
    """
    xyz = points_velo[:, :3]
    n = xyz.shape[0]
    xyz_h = np.hstack([xyz, np.ones((n, 1))])

    xyz_cam = (calib["Tr_velo_to_cam"] @ xyz_h.T).T
    xyz_rect = (calib["R0_rect"] @ xyz_cam.T).T

    in_front = xyz_rect[:, 2] > 0
    xyz_rect = xyz_rect[in_front]
    original_indices = np.where(in_front)[0]

    xyz_rect_h = np.hstack([xyz_rect, np.ones((xyz_rect.shape[0], 1))])
    uvw = (calib["P2"] @ xyz_rect_h.T).T
    uv = uvw[:, :2] / uvw[:, 2:3]
    depth = xyz_rect[:, 2]

    return uv, depth, original_indices





# Exercise 4.1 Keep only the points inside the bounding box

A naive approach to 3D segementation is just to take all the points that exist withing the 2D bounding box in the image.

Complete the function below to only keep the lidar points that fall withing the pedestrian bounding box
The cell below the function definition can help you visualize those points in the image

In [ ]:
def points_inside_2d_bbox(points, uv, front_indices, inside_image, bbox):
    """
    Return original Velodyne points whose projected uv coordinates are inside bbox.

    IMPORTANT: use points[front_indices[inside_image]], not points[:len(uv)].
    """
    left, top, right, bottom = bbox

    uv_valid = uv[inside_image]
    points_valid = points[front_indices[inside_image]]

    in_bbox = (
        (uv_valid[:, 0] >= left) & (uv_valid[:, 0] <= right) &
        (uv_valid[:, 1] >= top) & (uv_valid[:, 1] <= bottom)
    )

    return points_valid[in_bbox], uv_valid[in_bbox]

In [ ]:
# CELL REF: C4.2a
# Choose a KITTI sample that contains a pedestrian and load image/LiDAR/calibration

PEDESTRIAN_CLASSES = {"Pedestrian", "Person_sitting"}

sample_id_lidar = None
for sid in sample_ids:
    objs = parse_kitti_labels_full(sid)
    if any(obj["type"] in PEDESTRIAN_CLASSES for obj in objs):
        vel_path = os.path.join(VELODYNE_DIR, sid + ".bin")
        if os.path.exists(vel_path):
            sample_id_lidar = sid
            break

if sample_id_lidar is None:
    raise ValueError("No sample with both a pedestrian label and a Velodyne file was found.")

image_lidar = load_image(sample_id_lidar)
objects_lidar = parse_kitti_labels_full(sample_id_lidar)
points_raw = read_velodyne_bin(sample_id_lidar)
calib_lidar = read_calib(sample_id_lidar)

pedestrian_objects = [obj for obj in objects_lidar if obj["type"] in PEDESTRIAN_CLASSES]
obj = pedestrian_objects[0]
left, top, right, bottom = obj["bbox"]

uv, depth, front_indices = project_velodyne_to_image(points_raw, calib_lidar)
h, w = image_lidar.shape[:2]
inside_image = (
    (uv[:, 0] >= 0) & (uv[:, 0] < w) &
    (uv[:, 1] >= 0) & (uv[:, 1] < h)
)

bbox_points, bbox_uv = points_inside_2d_bbox(points_raw, uv, front_indices, inside_image, obj["bbox"])

print("Selected sample:", sample_id_lidar)
print("Image shape:", image_lidar.shape)
print("Raw point cloud:", points_raw.shape)
print("Selected object:", obj["type"])
print("2D bbox:", obj["bbox"].round(1).tolist())
print("LiDAR points inside pedestrian 2D bbox:", len(bbox_points))


In [ ]:
# CELL REF: C4.3a
# Visualise projected LiDAR points inside the pedestrian 2D box

fig, ax = plt.subplots(figsize=(14, 5))
ax.imshow(image_lidar)

ax.add_patch(patches.Rectangle((left, top), right-left, bottom-top, fill=False, linewidth=2, edgecolor="lime"))

if len(bbox_uv) > 0:
    ax.scatter(bbox_uv[:, 0], bbox_uv[:, 1], s=20, c="red")

ax.set_title("LiDAR points inside pedestrian 2D bounding box")
ax.axis("off")
plt.show()


In [ ]:
# CELL REF: C4.3b
# Visualise LiDAR points inside the pedestrian box in 3D

import open3d as o3d
import numpy as np



if len(bbox_points) == 0:
    raise ValueError("bbox_points is empty.")


bbox_pcd = o3d.geometry.PointCloud()

bbox_pcd.points = o3d.utility.Vector3dVector(
    bbox_points[:, :3]
)

# color points red
bbox_pcd.paint_uniform_color([1.0, 0.0, 0.0])


bbox_3d = bbox_pcd.get_axis_aligned_bounding_box()
bbox_3d.color = [0.0, 1.0, 0.0]


frame = o3d.geometry.TriangleMesh.create_coordinate_frame(
    size=1.0,
    origin=[0, 0, 0]
)


print("Opening Open3D viewer...")
print("Red = LiDAR points inside pedestrian 2D box")
print("Green = 3D bounding box")

o3d.visualization.draw_geometries(
    [bbox_pcd, bbox_3d, frame],
    window_name="3D LiDAR Points Inside Pedestrian Box"
)

**Q4.1**: What is the problem when using a naive approach of just taking all the points inside the bounding box?

YOUR ANSWER HERE

# Exercise 4.2 Use DBSCAN to cluster pedestrians

As you saw from the cells above just using the points inside the bounding box means extra points in the foreground and background are being kept that's wjy you need a clustering algorithm to keep only the points that actually belong to the pedestrian. For this we usethe DBSCAN algorithm information about the algorithm can be found [here](https://en.wikipedia.org/wiki/DBSCAN) and information about it's sklearn implementation can be found [here](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html).

In [ ]:
# CELL REF: C4.4a
# Automatic pedestrian cluster selection

def automatic_pedestrian_cluster(bbox_points, bbox_uv, bbox, eps=0.30, min_samples=3):
    if len(bbox_points) == 0:
        raise ValueError("No LiDAR points inside this 2D bbox.")

    if len(bbox_points) < min_samples:
        print("Not enough points for DBSCAN; returning all bbox points.")
        return bbox_points[:, :3], bbox_uv, np.zeros(len(bbox_points), dtype=int), 0

    xyz = bbox_points[:, :3]
    cluster_labels = DBSCAN(eps=eps, min_samples=min_samples, algorithm="kd_tree", n_jobs=-1).fit_predict(xyz)

    left, top, right, bottom = bbox
    box_w = right - left
    box_h = bottom - top
    box_cx = (left + right) / 2

    best_cluster = None
    best_score = -np.inf

    print("Cluster scores:")
    for c in np.unique(cluster_labels):
        if c == -1:
            continue

        cuv = bbox_uv[cluster_labels == c]
        cpts = xyz[cluster_labels == c]
        if len(cuv) < 3:
            continue

        u = cuv[:, 0]
        v = cuv[:, 1]

        c_w = np.max(u) - np.min(u)
        c_h_img = np.max(v) - np.min(v)
        c_h_3d = np.max(cpts[:, 2]) - np.min(cpts[:, 2])
        cx = np.mean(u)

        center_dist = abs(cx - box_cx) / max(box_w, 1)
        vertical_coverage = c_h_img / max(box_h, 1)
        mid_body_points = np.mean((v > top + 0.20 * box_h) & (v < top + 0.90 * box_h))
        bottom_only_penalty = np.mean(v > top + 0.90 * box_h)
        width_ratio = c_w / max(box_w, 1)
        size_score = np.log(len(cuv) + 1)

        score = (
            3.0 * mid_body_points +
            2.5 * vertical_coverage +
            0.8 * size_score +
            0.8 * min(c_h_3d, 2.0) -
            2.0 * center_dist -
            3.0 * bottom_only_penalty -
            0.8 * width_ratio
        )

        print(
            f"  cluster {c}: points={len(cuv):3d}, score={score:6.3f}, "
            f"vertical={vertical_coverage:.2f}, midbody={mid_body_points:.2f}, "
            f"bottom={bottom_only_penalty:.2f}, h3d={c_h_3d:.2f}"
        )

        if score > best_score:
            best_score = score
            best_cluster = c

    if best_cluster is None:
        print("No valid DBSCAN cluster found; returning all bbox points.")
        return xyz, bbox_uv, cluster_labels, None

    keep = cluster_labels == best_cluster
    return xyz[keep], bbox_uv[keep], cluster_labels, best_cluster


pedestrian_points, pedestrian_uv, cluster_labels, selected_cluster = automatic_pedestrian_cluster(
    bbox_points=bbox_points,
    bbox_uv=bbox_uv,
    bbox=obj["bbox"],
    eps=0.30,
    min_samples=3,
)

print("Selected cluster:", selected_cluster)
print("Pedestrian cluster points:", len(pedestrian_points))


**Q4.2**: Using the provided documentation as well as changing the parameters given to the pedestrian detection function how does each parameter affect the clusttering?

YOUR ANSWER HERE

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Convert image safely
image_show = np.array(image_lidar)

print("image shape:", image_show.shape)
print("bbox_uv shape:", bbox_uv.shape if bbox_uv is not None else None)
print("pedestrian_uv shape:", pedestrian_uv.shape if pedestrian_uv is not None else None)
print("bbox:", left, top, right, bottom)

fig, ax = plt.subplots(figsize=(14, 5))
ax.imshow(image_show)

ax.add_patch(
    patches.Rectangle(
        (left, top),
        right - left,
        bottom - top,
        fill=False,
        linewidth=2,
        edgecolor="lime"
    )
)

if bbox_uv is not None and len(bbox_uv) > 0:
    ax.scatter(
        bbox_uv[:, 0],
        bbox_uv[:, 1],
        s=20,
        alpha=0.4,
        c="yellow",
        label="all bbox LiDAR points"
    )
else:
    print("bbox_uv is empty")

if pedestrian_uv is not None and len(pedestrian_uv) > 0:
    ax.scatter(
        pedestrian_uv[:, 0],
        pedestrian_uv[:, 1],
        s=80,
        c="red",
        label="selected pedestrian cluster"
    )
else:
    print("pedestrian_uv is empty")

ax.set_xlim(0, image_show.shape[1])
ax.set_ylim(image_show.shape[0], 0)

ax.set_title("Automatically selected pedestrian LiDAR cluster")
ax.legend(loc="lower right")
ax.axis("off")
plt.show()

# Exercise 4.3 Create a 3d bounding box around the pedestrian

Finally it is time to create a 3d bounding box around the pedestrian.

In [ ]:
# CELL REF: C4.6a
# Add the pedestrian 3D bounding box to the raw point cloud using Open3D only

import open3d as o3d
import numpy as np

if len(pedestrian_points) == 0:
    raise ValueError("pedestrian_points is empty; run the clustering cell first.")


raw_pcd = o3d.geometry.PointCloud()
raw_pcd.points = o3d.utility.Vector3dVector(points_raw[:, :3])
raw_pcd.paint_uniform_color([0.6, 0.6, 0.6])


ped_pcd = o3d.geometry.PointCloud()
ped_pcd.points = o3d.utility.Vector3dVector(pedestrian_points[:, :3])
ped_pcd.paint_uniform_color([1.0, 0.0, 0.0])


pedestrian_bbox_3d = ped_pcd.get_axis_aligned_bounding_box()
pedestrian_bbox_3d.color = [0.0, 1.0, 0.0]


frame = o3d.geometry.TriangleMesh.create_coordinate_frame(
    size=1.0,
    origin=[0, 0, 0]
)


print("Opening Open3D viewer...")
print("Gray = raw point cloud")
print("Red = pedestrian points")
print("Green = pedestrian 3D bounding box")

o3d.visualization.draw_geometries(
    [raw_pcd, ped_pcd, pedestrian_bbox_3d, frame],
    window_name="Raw Point Cloud + Pedestrian 3D Bounding Box"
)